# <center>Data Cleaning and Feature Engineering Guide</center>

## <center>Overview</center>
This guide outlines the systematic approach to prepare the airport operations dataset for machine learning modeling. Follow these steps sequentially to ensure data quality and create meaningful features that will improve model performance.

---

## <center>Prerequisites</center>

Before starting, ensure you have:
- The raw dataset loaded into your working environment
- Understanding of the 29 original features
- Familiarity with the EDA findings (17,545 records, date range 2019-2021)

---

## <center>Phase 1: Data Cleaning</center>

### <center>Step 1: Remove Unnecessary Columns</center>

**Columns to remove:** `Date`, `Heavy_Flow`, `Continuous_Flows`, `line_type`, `end_time`

**Rationale:**
- **Date**: Redundant because temporal information will be extracted into separate features
- **Heavy_Flow**: Sparse binary flag identified in EDA with limited predictive value
- **Continuous_Flows**: Contains >90% missing data, making it unreliable for modeling
- **line_type**: Provides limited variance or contains redundant information already captured elsewhere
- **end_time**: Will be used temporarily for feature engineering, then removed

**Expected outcome:** Dataset reduced from 29 to 24 features

---

### <center>Step 2: Remove Duplicate Rows</center>

**Action:** Identify and remove the 1 duplicate row found during EDA (0.006% of data)

**Rationale:** Even a single duplicate can introduce bias in model training and validation, particularly in small subsets of data

**Verification:** After removal, confirm dataset has 17,544 records

---

### <center>Step 3: Encode the `queue_agent` Column</center>

**Action:** Transform the `queue_agent` column from text/null format to binary numeric

**Encoding rule:**
- Replace 'X' with 1 (agent present)
- Replace null/NaN values with 0 (no agent)

**Rationale:** Machine learning algorithms require numeric inputs. This binary encoding preserves the information while making it model-compatible

**Optional:** Convert to boolean data type for memory efficiency

---

### <center>Step 4: Encode Additional Binary Flag Columns</center>

**Columns to encode:** `pax_left`, `flight_cancelled`, `carousell_stopped`

**Encoding rule (same as Step 3):**
- 'X' → 1 (event occurred)
- null/NaN → 0 (event did not occur)

**Rationale:** These sparse binary indicators represent important operational events. Converting them to numeric format ensures they can be used as features while maintaining their interpretability

**Expected outcome:** Four boolean/binary columns ready for modeling

---

## <center>Phase 2: Feature Engineering</center>

### <center>Step 5: Create Time Period Categories</center>

**Action:** Create a new categorical column called `time_period` based on the `start_time` column

**Time period definitions:**
- **Early Morning**: 00:00 - 05:59
- **Morning**: 06:00 - 11:59  
- **Afternoon**: 12:00 - 17:59
- **Evening**: 18:00 - 23:59

**Rationale:** 
- Airport operations vary significantly by time of day
- Passenger volumes, staffing levels, and processing times follow clear temporal patterns
- This categorical feature captures these patterns more effectively than raw timestamps

**Implementation:** Extract the hour component from `start_time` and map it to the appropriate category

---

### <center>Step 6: Apply One-Hot Encoding to Categorical Variables</center>

**Columns to encode:** `Sector` and `Airline`

**Sector encoding:**
- Original: 2 categories (Domestic, International)
- After encoding: 1 binary column (use drop_first to avoid redundancy)
  - Example: `Sector_International` where 1 = International, 0 = Domestic

**Airline encoding:**
- Original: 13 airline categories
- After encoding: 12 binary columns (use drop_first to avoid multicollinearity)
  - Example: `Airline_2`, `Airline_3`, etc.

**Rationale:**
- Most ML algorithms cannot directly process categorical text data
- One-hot encoding creates binary features for each category
- Dropping the first category prevents the "dummy variable trap" (perfect multicollinearity)

**Expected outcome:** Replace 2 categorical columns with 13 binary columns (1 for Sector + 12 for Airlines)

---

### <center>Step 7: Calculate Event Duration Features</center>

**Action:** Create two new duration features measured in minutes

**Duration 1 - Line Duration:**
- Calculate the time difference between `Start_Line` and `End_Line`
- Name the new column: `line_duration_min`
- Represents: How long passengers spent in the queue

**Duration 2 - Service Duration:**
- Calculate the time difference between `Service_Start` and `Service_End`
- Name the new column: `service_duration_min`
- Represents: How long the actual service/processing took

**Rationale:**
- Duration metrics are more meaningful than raw timestamps for prediction
- They capture operational efficiency and resource utilization patterns
- Shorter durations may indicate better performance or adequate staffing

**Important:** Convert results to minutes and round to 2 decimal places for consistency

---

### <center>Step 8: Drop Timestamp Columns After Feature Creation</center>

**Columns to remove:** `Start_Line`, `End_Line`, `Service_Start`, `Service_End`

**Rationale:**
- These raw timestamp columns have been transformed into meaningful duration features
- Keeping them would add unnecessary dimensionality without additional predictive value
- Raw timestamps can cause data leakage in time-series predictions

**Expected outcome:** Four columns removed, replaced by two engineered duration features

---

### <center>Step 9: Calculate Total Checked Bags</center>

**Action:** Create a new column called `total_checked_bags`

**Calculation:** Sum of `Luggage_In` + `Luggage_Out`

**Rationale:**
- Provides an aggregate view of baggage handling volume
- The strong correlation (0.769) between Luggage_In and Luggage_Out suggests they represent related aspects of the same process
- Total bags can be a better predictor of resource needs than individual components

**Note:** Keep the original `Luggage_In` and `Luggage_Out` columns as they may still provide value independently

---

### <center>Step 10: Drop Year Column</center>

**Action:** Remove the `Year` column from the dataset

**Rationale:**
- Year information is already captured through other temporal patterns in the data
- Keeping Year as a single numeric feature may introduce unintended temporal bias
- The dramatic variation between years (2020 COVID impact) might be better handled through specific indicators
- Alternative approaches (like pandemic flags) can capture the 2020 anomaly more explicitly

**Consideration:** If you created a pandemic indicator in a later step, Year becomes redundant

---

## <center>Phase 3: Outlier Detection and Handling</center>

### <center>Step 11: Check and Remove Outliers</center>

**Action:** Systematically identify and handle extreme values in numerical columns

**Target columns for outlier analysis:**
- `Pax_In_Line` (EDA showed max=65, mean=4.2)
- `PAX_Served` (max=12, mean=1.8)
- `Counters` (max=15, mean=5.4)
- `Luggage_In` (max=25, mean=3.0)
- `Luggage_Out` (max=15, mean=1.7)

**Detection method - IQR (Interquartile Range):**
1. Calculate Q1 (25th percentile) and Q3 (75th percentile)
2. Calculate IQR = Q3 - Q1
3. Define outlier bounds:
   - Lower bound = Q1 - 1.5 × IQR
   - Upper bound = Q3 + 1.5 × IQR
4. Flag values outside these bounds

**Handling strategy - Choose based on context:**

**Option A - Remove outliers:**
- Best for: Clear data entry errors or impossible values
- Caution: Some "outliers" may represent legitimate peak events (holiday travel, special events)
- Recommendation: Only remove if outliers represent <5% of data AND seem invalid

**Option B - Cap outliers (Winsorization):**
- Best for: Legitimate extreme values that could skew models
- Method: Replace values beyond 1st/99th percentiles with the percentile values
- Preserves data volume while reducing extreme influence

**Option C - Keep and flag:**
- Best for: Peak events that are real and important
- Method: Create binary indicators like `is_peak_volume`
- Allows models to learn these patterns separately

**Important considerations:**
- Airport operations naturally have peak periods (holidays, connecting flights)
- Values like 65 passengers in line or 25 pieces of luggage might be real surge events
- Investigate outliers before removing—look for patterns by date, airline, or sector
- Document your decision and reasoning for reproducibility

**Expected outcome:** Cleaner dataset with justified handling of extreme values, typically removing <2-5% of records if Option A is chosen

---

### <center>Step 12: Handle Zero Counter Records</center>

**Action:** Investigate and address records where `Counters = 0`

**Investigation steps:**
1. Count how many records have zero counters
2. Check if these records have other valid data (PAX_Served, luggage counts)
3. Look for patterns—do they occur on specific dates, airlines, or times?

**Possible interpretations:**
- **Data entry error**: Counter information was not recorded
- **Special process**: Some operations may not use traditional counters
- **Cancelled or aborted operations**: Event started but didn't proceed normally

**Handling decision tree:**

**If very few records (<10):**
- Remove them as likely data quality issues
- Impact on dataset is minimal

**If substantial records (>10):**
- **And they have valid other data**: Keep them but create a flag column `zero_counter_flag`
- **And they're mostly missing other data too**: Consider removing as incomplete records
- **If they show a pattern**: Investigate further—might represent valid operational scenario

**Expected outcome:** Either cleaned dataset without zero-counter records, or flagged records for model awareness

---

### <center>Step 13: Handle 2020 Pandemic Period Data</center>

**Action:** Address the COVID-19 impact visible in the data (2020 has only 12.8% vs 59% in 2019)

**Context:**
- 2020 data represents an anomalous operational period
- Passenger volumes, flight frequencies, and operational patterns were dramatically different
- These patterns may not be representative of normal operations

**Handling strategies:**

**Strategy A - Create pandemic indicator (Recommended):**
- Add a binary column `is_pandemic_period` where 2020 = 1, others = 0
- Allows models to learn that 2020 operated under different conditions
- Keeps all data while acknowledging the special circumstances

**Strategy B - Separate modeling approach:**
- Train separate models for pre-pandemic (2019), pandemic (2020), and post-pandemic (2021)
- Use when operational conditions are fundamentally different
- Better for scenario-specific predictions

**Strategy C - Remove 2020 data:**
- Use only if predicting "normal" operations
- Loses valuable data but ensures training on typical patterns
- Consider if current operations have returned to pre-pandemic norms

**Strategy D - Weight-based approach:**
- Keep all data but apply lower weights to 2020 records during model training
- Balanced approach that uses the data without letting it dominate

**Recommendation:** Start with Strategy A (pandemic indicator) as it preserves data while capturing the contextual difference

**Expected outcome:** Dataset prepared to handle temporal anomalies appropriately

---

## <center>Phase 4: Final Validation</center>

### <center>Step 14: Comprehensive Data Quality Checks</center>

**Action:** Perform final verification before proceeding to modeling

**Checklist:**

**1. Missing Values Audit:**
- Check every column for null/NaN values
- Verify that all binary encodings were successful
- Confirm no unexpected missing data was introduced during transformations

**2. Data Type Verification:**
- Numeric columns are int or float
- Binary flags are boolean or int (0/1)
- One-hot encoded columns are int (0/1)
- No unexpected object/string types remain

**3. Shape Validation:**
- Count final number of rows (should be ~17,544 or fewer if outliers removed)
- Count final number of columns
- Compare to expected dimensions based on transformations

**4. Duplicate Check:**
- Verify no duplicates remain
- Especially important after all transformations

**5. Value Range Validation:**
- Confirm all values are within expected ranges
- Binary columns only contain 0 and 1
- Negative values only appear where logically valid
- Check for any infinite values

**6. Consistency Checks:**
- Duration features are positive (end time > start time)
- Total_checked_bags = Luggage_In + Luggage_Out
- Counter counts are reasonable (typically 1-15)

**7. Statistical Sanity Checks:**
- Recalculate basic statistics (mean, median, std)
- Compare to original EDA to ensure transformations didn't corrupt data
- Verify distributions still make operational sense

**Expected outcome:** A validation report confirming the dataset is clean, consistent, and ready for modeling

---

## <center>Save the Cleaned Dataset</center>

**Action:** Export the final cleaned and engineered dataset

**File format recommendations:**
- **CSV**: Universal compatibility, human-readable
- **Parquet**: Better performance, preserves data types, smaller file size

**Naming convention:** Use descriptive names with version/date
- Example: `airport_data_cleaned_v1_2026-01-28.csv`

**Documentation:** Save a summary file alongside your data containing:
- Original dataset shape and final shape
- List of all transformations applied
- Outlier handling decisions
- Number of records removed and reasons
- Date of processing

**Backup:** Keep original raw data untouched in separate location

---

## <center>Summary of Transformations</center>

### Columns Removed (9 total):
1. Date
2. Heavy_Flow
3. Continuous_Flows
4. line_type
5. end_time
6. Start_Line
7. End_Line
8. Service_Start
9. Service_End
10. Year

### Columns Encoded (4 total):
1. queue_agent (X → 1, null → 0)
2. pax_left (X → 1, null → 0)
3. flight_cancelled (X → 1, null → 0)
4. carousell_stopped (X → 1, null → 0)

### Columns One-Hot Encoded (2 original → ~13 new):
1. Sector (2 categories → 1 binary column)
2. Airline (13 categories → 12 binary columns)

### New Features Created (4 total):
1. time_period (4 categories: Early_Morning, Morning, Afternoon, Evening)
2. line_duration_min (calculated from Start_Line and End_Line)
3. service_duration_min (calculated from Service_Start and Service_End)
4. total_checked_bags (sum of Luggage_In and Luggage_Out)

### Data Quality Actions:
- Removed 1 duplicate row
- Handled outliers in 5 numerical columns
- Addressed zero counter records
- Handled 2020 pandemic period data

### Final Dataset Characteristics:
- **Records**: ~17,544 (or fewer after outlier removal)
- **Features**: Approximately 35-40 (depending on encoding choices)
- **Memory**: Reduced through data type optimization
- **Quality**: Validated and model-ready

---

## <center>Next Steps After Cleaning</center>

Once data cleaning is complete, proceed with:

1. **Exploratory Analysis on Cleaned Data**
   - Verify distributions after transformations
   - Check correlations with new engineered features
   - Identify potential multicollinearity issues

2. **Feature Selection**
   - Use correlation analysis to identify redundant features
   - Apply feature importance techniques
   - Consider dimensionality reduction if needed

3. **Feature Scaling**
   - Apply StandardScaler for algorithms sensitive to scale
   - Or MinMaxScaler for bounded ranges
   - Don't scale one-hot encoded columns

4. **Train-Test Split**
   - Use temporal split to respect time ordering
   - Consider stratification by Sector or Airline
   - Typical split: 70-80% train, 20-30% test

5. **Model Development**
   - Start with baseline models (Linear Regression, Decision Trees)
   - Progress to ensemble methods (Random Forest, XGBoost)
   - Use cross-validation appropriate for time-series data

---

## <center>Important Reminders</center>

- **Document everything**: Keep notes on every decision made during cleaning
- **Version control**: Save intermediate datasets at key transformation points
- **Reproducibility**: Your process should be repeatable with clear steps
- **Domain knowledge**: Consult airport operations experts when uncertain about outliers
- **Iterative process**: You may need to revisit cleaning after initial modeling
- **Validation**: Always validate transformations with sample checks

---

# <center>**End of Guide**</center>